# Laboratorio 2
---

**Grupo**: 8

**Miembros del Grupo**: 

* Bedón, Bruno
* Galve Mateo, Carlos (Lider)
* Sanchez Mejía, Carlos
* Giron Rene, Omar Andony
* Avalos, Wilder

In [ ]:
# Libreiras
import os

import pandas as pd
from agent_framework import (
    Agent,
    CharacterEstimatorTokenizer,
    InMemoryHistoryProvider,
    Message,
    apply_compaction,
)
from agent_framework.openai import OpenAIChatClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

from dsc.constants.common import PROJECT_ROOT
from dsc.evaluation.agent import valid_agent
from dsc.prompts.utils import build_prompt_from_template
from dsc.strategies.compaction import compaction_strategy
from dsc.utils.common import read_yaml

# Variables de entorno
load_dotenv()
endpoint_llm = os.getenv("FOUNDRY_MODEL_ENDPOINT")
model_llm = os.getenv("FOUNDRY_MODEL_DEPLOYMENT")
version_llm = os.getenv("FOUNDRY_MODEL_VERSION")

# Confguración de visualización de pandas
pd.set_option("display.max_colwidth", None)

### Construcción del agente

##### A. Definir las instrucciones del agente

La configuración del system prompt está en `dsc/config/system_prompt.yaml`, donde se define rol, objetivo, contexto, restricciones y formato de salida.

Esta configuración es aplicada a la plantilla que se encuentra las constantes definidas en `dsc/src/dsc/constants/plantillas.py`

**¿Qué técnicas de prompting utiliza su agente y por qué son necesarias para su caso de uso?**

**1. Asignación de roles:** El agente actúa como un asesor bancario especializado en soluciones de pago para préstamos efectivos, con un perfil empático, claro y transparente. Es necesaria porque los clientes suelen llegar preocupados o avergonzados por sus deudas. El rol orienta al modelo a priorizar una salida sostenible para el cliente y a evitar un lenguaje de cobranza agresivo.

**2. Zero-shot con instrucciones detalladas:** El prompt define con precisión el objetivo, el contexto, las restricciones y el formato de salida. Esto es fundamental en un entorno bancario regulado, porque limita el alcance a préstamos efectivos, prohíbe inventar tasas o montos, impide solicitar datos sensibles y evita garantizar aprobaciones.

**3. Few-shot:** Incluimos ejemplos de diálogo para que el modelo imite el tono esperado, respete la regla de máximo dos preguntas por mensaje y sepa cómo responder consultas fuera de alcance. Los ejemplos transmiten el estilo y el criterio mejor que las reglas por sí solas.

**4. Chain of Thought (interno):** Antes de recomendar, el agente analiza paso a paso la capacidad de pago del cliente (ingresos menos gastos), la compara con la cuota actual y descarta las opciones inviables. El razonamiento no se muestra al cliente, solo la conclusión. Esto asegura recomendaciones coherentes con la situación real del cliente.

**5. ReAct (siguiente etapa):** Cuando el agente se conecte a herramientas como un simulador de cuotas o la consulta de saldo, combinará razonamiento y acciones. Así obtendrá datos reales en lugar de suponerlos, lo que refuerza la restricción de no inventar información.

**Técnica descartada: Self-consistency.** Generar múltiples respuestas por cada mensaje aumentaría el costo y la latencia, lo que no es viable en una conversación en tiempo real. Podría usarse fuera de producción para evaluar la calidad de las recomendaciones del agente.

##### B. Probar el alcance del agente

In [2]:
# Credenciales
credential = DefaultAzureCredential()

# Se instacia el cliente de OpenAI con el modelo y las credenciales
client = OpenAIChatClient(
    model=model_llm,  # Nombre exacto del deployment
    base_url=endpoint_llm,
    credential=credential,
)

In [3]:
# -----------------------------------------------------------------------------
# Se crea el system prompt a partir de la plantilla y el archivo de configuración
config_system_prompt = read_yaml(PROJECT_ROOT / "config/system_prompt.yaml")
system_prompt = build_prompt_from_template(config_system_prompt)

# -----------------------------------------------------------------------------
# Se crea el agente
agente = Agent(
    client=client,
    name="BancoCuyDeOroAsesorSdP",
    instructions=system_prompt
)

**Verifiquen que el agente mantenga la especialidad definida y no responda consultas fuera de su alcance.**

In [4]:
resultados_validacion = await valid_agent(agente=agente, verbose=False)
resultados_validacion

,categoria,mensaje,respuesta
0,especialidad,"Me atrasé dos cuotas de mi préstamo personal, ¿qué puedo hacer?","Entiendo que esta situación puede preocuparte, pero podemos revisar alternativas para regularizar tu préstamo sin asumir una cuota que no puedas sostener. ¿Cuál es el monto de tu cuota mensual y la causa principal del atraso (por ejemplo, reducción de ingresos o un gasto imprevisto)?"
1,especialidad,¿Qué diferencia hay entre reprogramar y refinanciar?,"La diferencia principal es cómo se modifica tu préstamo:\n\n- **Reprogramación:** ajusta las condiciones del préstamo vigente, por ejemplo, ampliando el plazo, cambiando la fecha de pago o incorporando un periodo de gracia. Puede reducir la cuota mensual, pero normalmente aumenta el tiempo de pago y los intereses totales. Si estás al día, por lo general no implica cancelar el préstamo original; el efecto en tu historial depende de la evaluación y de las condiciones aplicadas.\n\n- **Refinanciamiento:** reemplaza el préstamo actual por uno nuevo, con nuevas condiciones de cuota, plazo y tasa. Puede incluir cuotas vencidas u otras deudas que se consoliden, si la entidad lo ofrece. También puede reducir la cuota, pero al ampliar el plazo podría aumentar el costo total; además, el nuevo contrato y el historial de pagos previos pueden influir en tu calificación crediticia.\n\nAmbas opciones están sujetas a evaluación. Para orientarte mejor, ¿estás al día o tienes cuotas atrasadas?"
2,especialidad,"Tengo tres tarjetas y un préstamo, ¿puedo juntarlos en una sola cuota?","Sí, puedes solicitar una **consolidación de deudas**, que reúne tus tres tarjetas y tu préstamo en un solo préstamo efectivo con una cuota. La aprobación, tasa y plazo dependen de la evaluación del banco; además, una cuota menor podría implicar pagar más intereses en total y las tarjetas consolidadas podrían quedar sujetas a condiciones específicas.\n\nPara orientarte mejor, ¿cuánto debes aproximadamente en cada tarjeta y en el préstamo, y cuánto pagas mensualmente entre todos?"
3,especialidad,¿Me garantizas que me aprueban la reprogramación?,"No puedo garantizarte la aprobación. La reprogramación está sujeta a la evaluación crediticia del banco y a las condiciones de tu préstamo.\n\nSí puedo ayudarte a revisar si parece una alternativa viable según tu cuota actual, saldo, días de atraso e ingresos y gastos mensuales. Si me compartes esos datos aproximados —sin claves, códigos ni números completos— te orientaré sobre las opciones disponibles y sus posibles efectos."
4,fuera_alcance,¿Quién ganó el último partido de la selección?,"No puedo ayudar con resultados deportivos. Sí puedo orientarte sobre soluciones de pago para préstamos efectivos, como reprogramación, refinanciamiento o convenios de pago."
5,fuera_alcance,Dame una receta de lomo saltado,"No puedo ayudarte con recetas de cocina. Sí puedo orientarte sobre soluciones de pago para préstamos efectivos, como reprogramación, refinanciamiento o convenios de pago."
6,fuera_alcance,Escríbeme un script en Python para ordenar una lista,"No puedo ayudarte con programación; soy un asesor de soluciones de pago para préstamos efectivos. Puedo orientarte sobre atrasos, reprogramación, refinanciamiento, consolidación de deudas, cambio de fecha de pago o convenios de pago."
7,otra_especialidad,Quiero abrir una cuenta de ahorros,"La apertura de una cuenta de ahorros corresponde al canal especializado de cuentas. Puedes realizarla desde la app móvil, en la web www.bancocuydeoro.pe o acercándote a cualquier agencia. \n\nAquí puedo ayudarte con soluciones de pago para préstamos efectivos, como reprogramaciones, refinanciamientos o convenios de pago."
8,otra_especialidad,Me aparece un cargo que no reconozco en mi tarjeta,"Si no reconoces un cargo, podría tratarse de un fraude. Bloquea de inmediato tu tarjeta desde la app o llamando al **(01) 000-1111**; luego reporta el caso a la **Línea de Fraudes: (01) 000-2222** o en cualquier agencia. No compartas claves, códigos de ver

### Memoria de corto plazo

In [5]:
# Manejo de memoria utilizando Agent Framework
agente = Agent(
    client=client,
    name="BancoCuyDeOroAsesorSdP",
    instructions=system_prompt,
    default_options={
        "store": False
    },
    context_providers=[
        InMemoryHistoryProvider(
            "historial",
            load_messages=True
        )
    ],
)

# Crear una sesión de conversación.
sesion = agente.create_session()

In [6]:

# Primera interacción.
user_prompt1 = "Hola, tengo un préstamo personal y me atrasé 15 días en mi cuota."
respuesta1 = await agente.run(user_prompt1, session=sesion)
print(respuesta1)

Hola, gracias por contármelo; un atraso de 15 días puede pasar y podemos revisar una salida sostenible sin juzgarte. ¿Cuál es el monto de tu cuota y puedes pagarla completa o parcialmente en los próximos días? ¿El atraso se debe a una reducción de ingresos, un gasto imprevisto u otra razón?


In [7]:
# Segunda interacción: se reutiliza la misma sesión.
user_prompt2 = "Mi cuota es de S/ 850 y me quedan 18 meses por pagar."
respuesta2 = await agente.run(user_prompt2, session=sesion)
print(respuesta2)

Entiendo: tu cuota actual es de **S/ 850** y te quedan **18 meses**; el atraso es de 15 días. Para estimar qué alternativa sería sostenible, ¿cuánto puedes destinar mensualmente al préstamo hoy y cuánto suman aproximadamente tus ingresos y gastos fijos?


In [8]:
user_prompt3 = "Me redujeron el sueldo: ahora gano S/ 3,000 y mis gastos fijos son S/ 2,500."
respuesta3 = await agente.run(user_prompt3, session=sesion)
print(respuesta3)

Gracias por compartirlo. Con ingresos de **S/ 3,000** y gastos fijos de **S/ 2,500**, tu capacidad disponible estimada es de **S/ 500**, menor que la cuota actual de **S/ 850**; por eso, mantenerla podría presionar tu presupuesto. Para comparar una reprogramación o refinanciamiento, ¿cuál es el saldo pendiente aproximado que aparece en tu estado de cuenta o app?


In [9]:
user_prompt4 = "Con todo lo que te conté, ¿cuánto podría pagar al mes y qué opción me conviene?"
respuesta4 = await agente.run(user_prompt4, session=sesion)
print(respuesta4)

### 1. Resumen de tu situación

Con ingresos de **S/ 3,000** y gastos fijos de **S/ 2,500**, tu disponibilidad mensual estimada es de **S/ 500**. La cuota de **S/ 850** supera esa capacidad en aproximadamente **S/ 350**, por lo que mantenerla podría generar nuevos atrasos.

### 2. Opciones

**1. Reprogramación del préstamo — opción más recomendable para evaluar**
- Consiste en modificar el cronograma para reducir la cuota y ampliar el plazo.
- **Cuota objetivo estimada:** entre **S/ 400 y S/ 500**; no convendría asumir una cuota mayor a tu disponibilidad actual.
- **Plazo:** sería mayor a los 18 meses restantes, pero no puedo calcularlo sin conocer el saldo pendiente y la tasa.
- **Ventaja:** podría hacer la cuota más manejable.
- **Desventaja:** al extender el plazo, normalmente aumenta el total de intereses pagados.
- **Historial:** el atraso de 15 días puede quedar registrado según las políticas de reporte; la modificación también podría reflejarse como un crédito reprogramado.

**2

In [10]:
user_prompt5 = "Cuál era mi cuota"
respuesta5 = await agente.run(user_prompt4, session=sesion)
print(respuesta5)

### 1. Resumen de tu situación

Con ingresos de **S/ 3,000** y gastos fijos de **S/ 2,500**, tu disponibilidad mensual estimada es de **S/ 500**. Como tu cuota actual es de **S/ 850**, existe una diferencia de **S/ 350** y ya tienes 15 días de atraso.

### 2. Opciones

**1. Reprogramación — la primera opción a evaluar**
- Ajusta el cronograma para reducir la cuota y ampliar el plazo.
- **Cuota referencial conveniente:** entre **S/ 400 y S/ 500**; idealmente cerca de S/ 400–450 para conservar un margen.
- **Plazo:** mayor a los 18 meses restantes; no es posible calcularlo sin el saldo y la tasa.
- **Costo:** al ampliar el plazo, normalmente pagarías más intereses en total.
- **Historial:** el atraso puede quedar registrado y la reprogramación podría reflejarse como una modificación del crédito, según las políticas de reporte.

**2. Refinanciamiento**
- Genera un nuevo cronograma para ordenar la deuda, posiblemente incluyendo la cuota vencida.
- **Cuota referencial:** debería mantenerse 

**¿Cuántas interacciones debería conservar la memoria de corto plazo de nuestro agente y por qué?**

Definimos conservar **20 interacciones** (20 mensajes del cliente y 20 respuestas del agente, es decir, 40 mensajes).

**Cálculo según el flujo definido en nuestro prompt:**

- **Recopilación de información (4 a 6 interacciones):** el objetivo exige conocer seis datos (saldo, cuota actual,
  días de atraso, causa de la dificultad, ingresos y gastos). Como la regla de formato limita al agente a una o dos
  preguntas por mensaje, se necesitan al menos 3 interacciones, y en la práctica más, porque el cliente suele
  responder de forma parcial y el paso 5 del razonamiento obliga a preguntar cualquier dato faltante antes de recomendar.
- **Presentación de alternativas (1 interacción):** el agente entrega la estructura completa: resumen, hasta tres
  opciones, recomendación, próximos pasos y aviso.
- **Dudas sobre las opciones (3 a 6 interacciones):** con hasta tres opciones, y cada una con cinco aspectos de impacto
  (nueva cuota, plazo, intereses totales, costo efectivo e historial crediticio), es esperable que el cliente pida
  aclaraciones o compare alternativas.
- **Próximos pasos y cierre (1 a 2 interacciones):** documentos, canal de atención y confirmación.
- **Desvíos (1 a 2 interacciones):** consultas mixtas o ambiguas que el agente debe aclarar o derivar sin perder el
  hilo del caso.

Esto suma entre 10 y 17 interacciones; con 20 dejamos margen para clientes que necesitan más explicaciones.

**Por qué no menos:** los datos clave se dan al inicio y el agente los necesita al final. El razonamiento interno
calcula la capacidad de pago (ingresos menos gastos) y la compara con la cuota actual para descartar opciones
inviables; si la memoria olvidara esos datos, la recomendación sería incorrecta o el agente volvería a preguntarlos,
lo que frustra a un cliente que muchas veces ya llega preocupado o avergonzado.

**Por qué no más:** nuestro system prompt ya es extenso (rol, restricciones, canales, razonamiento, formato y cuatro
ejemplos), y las respuestas con alternativas son largas, por lo que un historial mayor aumenta el costo y la latencia
sin mejorar la atención. Además, el historial contiene datos financieros personales; conservar solo lo necesario para
resolver el caso es coherente con nuestra restricción de cumplir la normativa de protección de datos personales.
Finalmente, una conversación que supera las 20 interacciones suele indicar que la situación supera las alternativas
disponibles, y nuestro prompt ya establece que esos casos deben derivarse a un asesor humano.

### Estrategia de compactación

**¿Qué estrategia de compactación utilizarán y por qué es adecuada para su agente?**

La estrategia elegida es `TokenBudgetComposedStrategy`, con un presupuesto de 6000 tokens y compuesta por dos etapas:
1. `SummarizationStrategy`
2. `TruncationStrategy` (respaldo)

**¿Por qué es adecuada para nuestro agente?**

El agente trabaja con `store: False` y un `InMemoryHistoryProvider`, por lo que el historial completo se reenvía al
modelo en cada turno. Sin control, el consumo de tokens, el costo y la latencia crecen con cada interacción.

- **Presupuesto en tokens:** `TokenBudgetComposedStrategy` mide el historial en tokens, la métrica que determina el
  costo de entrada y el límite de la ventana de contexto. Mientras el historial esté bajo el presupuesto no se
  modifica; al superarlo, se ejecutan las etapas en orden y, con `early_stop=True`, se detiene en cuanto vuelve a
  entrar en el límite. Esto estabiliza el costo por turno, aunque no reemplaza un control de gasto total (salida,
  número de llamadas y llamadas de resumen se gestionan aparte). Medir en tokens es especialmente útil en nuestro
  caso, porque las respuestas con alternativas de pago son mucho más largas que los mensajes de recopilación.
- **Dimensionamiento del presupuesto:** fijamos 6000 tokens a partir de una estimación de nuestro caso. El system
  prompt ocupa unos ~2,300 tokens, y el historial ya compactado (resumen más los 6 mensajes recientes, incluida la
  respuesta con alternativas, que es la más larga del flujo) ocupa unos ~2,000. Esto suma cerca de 4,300 tokens, por
  lo que el presupuesto de 6000 deja un margen para cubrir la imprecisión del estimador y evitar que
  la truncación se active en cada turno.
- **Preservar datos críticos:** el cliente entrega al inicio la información que el agente necesita al final para
  calcular su capacidad de pago y recomendar: saldo, cuota, plazo, días de atraso, causa de la dificultad, ingresos y
  gastos. Por eso la primera etapa es `SummarizationStrategy`, con un prompt que exige conservar los montos exactos,
  las alternativas presentadas y las decisiones del cliente, en lugar de simplemente borrar mensajes antiguos. La
  configuramos para mantener completos los mensajes más recientes y resumir solo los anteriores.
- **Respaldo basado en tokens:** si el resumen no basta (por ejemplo, mensajes recientes muy largos),
  `TruncationStrategy` elimina los grupos más antiguos hasta bajar del límite. Se prefiere sobre
  `SlidingWindowStrategy` porque esta conserva un número fijo de mensajes sin importar su tamaño, mientras que la
  truncación con tokenizer se adapta al peso real de los mensajes.
- **Estrategia descartada:** `ToolResultCompactionStrategy` no aplica por ahora, porque el agente no usa herramientas.
  La reconsideraremos al integrar un simulador de cuotas o la consulta de saldo.

**Riesgos considerados:**
- El resumen puede alterar o perder cifras; lo mitigamos con un prompt que prohíbe redondear o inventar montos y
  conservando completos los mensajes más recientes.
- La truncación elimina información sin resumirla, y podría eliminar el propio mensaje de resumen por ser de los más
  antiguos. Por eso solo actúa como respaldo, y el presupuesto de 6000 tokens deja margen suficiente sobre el
  historial compactado para que rara vez se active.
- El conteo de tokens es una estimación (`CharacterEstimatorTokenizer`), por lo que el presupuesto incluye margen y lo
  ajustaremos según las mediciones de conversaciones reales.

**Relación con la memoria de corto plazo:**

En la sección anterior definimos conservar 20 interacciones. Con la compactación, ese valor cambia de significado: ya
no indica cuántos mensajes se envían completos al modelo, sino cuánta información de la conversación debe seguir
disponible para el agente. La estrategia mantiene completos solo los 6 mensajes más recientes (3 interacciones) y
reemplaza los anteriores por un resumen que conserva los montos exactos, las alternativas presentadas y las
decisiones del cliente. Así el agente sigue teniendo la información de toda la conversación, mientras el historial
enviado se mantiene dentro de un presupuesto de 6000 tokens. Las 20 interacciones pasan a ser la extensión esperada
de una conversación: si se supera, lo interpretamos como un caso complejo que conviene derivar a un asesor humano.


**NOTA**: para las pruebas en este notebook, se ponen valores más pequeños en la configuración de la compactacion:
| Parámetro | Prueba | Producción |
|---|---|---|
| `token_budget` | 1000 tokens | 6000 tokens |
| `target_count` (SummarizationStrategy) | 4 mensajes | 6 mensajes |
| `threshold` (SummarizationStrategy) | 2 mensajes | 2 mensajes |
| `max_n` (TruncationStrategy) | 1200 tokens | 6200 tokens |
| `compact_to` (TruncationStrategy) | 1000 tokens | 6000 tokens |

In [11]:
agente = Agent(
    client=client,
    name="BancoCuyDeOroAsesorSdP",
    instructions=system_prompt,
    default_options={
        "store": False
    },
    context_providers=[
        InMemoryHistoryProvider(
            "historial",
            load_messages=True
        )
    ],
    compaction_strategy=compaction_strategy,
)
sesion = agente.create_session()

In [12]:
print("🤖 Agente: Hola 👋 ¿En qué tema de finanzas personales puedo ayudarte?")
print("Escribe 'salir' para finalizar.\n")

ls_test_msj_compaction = []
while True:

    pregunta = input("Escribe tu mensaje: ").strip()

    if not pregunta:
        continue

    print(f"👤 Usuario: {pregunta}")

    if pregunta.lower() in ["salir", "exit", "quit"]:
        print("🤖 Agente: ¡Hasta luego! 👋\n")
        break

    respuesta = await agente.run(
        pregunta,
        session=sesion
    )

    ls_test_msj_compaction.append((pregunta, respuesta.text))

    print(f"🤖 Agente: {respuesta}\n")

# Hola, tengo un préstamo personal con un saldo de S/ 12,400 y me atrasé 15 días.
# Mi cuota es de S/ 850 y me quedan 18 meses. Me redujeron el sueldo.
# Ahora gano S/ 3,000 al mes y mis gastos fijos son S/ 2,500.
# ¿Qué diferencia hay entre reprogramar y refinanciar?
# ¿Qué documentos necesito para solicitar una reprogramación?
# Antes de decidir, recuérdame: ¿cuál era mi saldo, mi cuota y cuánto gano?

🤖 Agente: Hola 👋 ¿En qué tema de finanzas personales puedo ayudarte?
Escribe 'salir' para finalizar.

👤 Usuario: Hola, tengo un préstamo personal con un saldo de S/ 12,400 y me atrasé 15 días.
🤖 Agente: Gracias por contármelo; entiendo que un atraso puede preocupar, pero todavía podemos revisar alternativas para regularizarlo. Para orientarte mejor, ¿cuál es tu cuota mensual actual y qué causó el atraso: una reducción de ingresos, un gasto imprevisto u otro motivo?

👤 Usuario: Mi cuota es de S/ 850 y me quedan 18 meses. Me redujeron el sueldo.
🤖 Agente: Gracias por precisarlo; la reducción de sueldo explica la dificultad y revisaremos una cuota sostenible, sin asumir que puedas mantener los S/ 850 actuales. ¿Cuánto recibes ahora al mes y cuánto suman aproximadamente tus gastos esenciales fijos, como vivienda, alimentación, servicios y otras deudas?

👤 Usuario: Ahora gano S/ 3,000 al mes y mis gastos fijos son S/ 2,500.
🤖 Agente: ### 1. Resumen de la situación  
Tienes un saldo aproxima

##### Prueba de compactación

In [13]:
ls_msj = []
i = 1
for pregunta, respuesta in ls_test_msj_compaction:
    print("#-------------------------------------------------------")
    print("Interacciones:", i)
    ls_msj.append(Message(role="user", contents=[pregunta]))
    ls_msj.append(Message(role="assistant", contents=[respuesta]))

    nu_msj_iniciales = len(ls_msj)

    print("=============> Mensajes de la conversación antes de la compactación:")
    for m in ls_msj:
        print(f"[{m.role}] {m.text}\n")

    ls_msj = await apply_compaction(
        messages=ls_msj,
        strategy=compaction_strategy,
        tokenizer=CharacterEstimatorTokenizer(),
    )

    print("=============> Mensajes de la conversación después de la compactación:")
    for m in ls_msj:
        print(f"[{m.role}] {m.text}\n")
    i += 1

    print(f"=============> Antes: {nu_msj_iniciales} mensajes | Después: {len(ls_msj)} mensajes\n")

#-------------------------------------------------------
Interacciones: 1
=============> Mensajes de la conversación antes de la compactación:
[user] Hola, tengo un préstamo personal con un saldo de S/ 12,400 y me atrasé 15 días.

[assistant] Gracias por contármelo; entiendo que un atraso puede preocupar, pero todavía podemos revisar alternativas para regularizarlo. Para orientarte mejor, ¿cuál es tu cuota mensual actual y qué causó el atraso: una reducción de ingresos, un gasto imprevisto u otro motivo?

=============> Mensajes de la conversación después de la compactación:
[user] Hola, tengo un préstamo personal con un saldo de S/ 12,400 y me atrasé 15 días.

[assistant] Gracias por contármelo; entiendo que un atraso puede preocupar, pero todavía podemos revisar alternativas para regularizarlo. Para orientarte mejor, ¿cuál es tu cuota mensual actual y qué causó el atraso: una reducción de ingresos, un gasto imprevisto u otro motivo?

=============> Antes: 2 mensajes | Después: 2 mens

In [14]:
print("🤖 Agente: Hola 👋 ¿En qué tema de finanzas personales puedo ayudarte?")
print("Escribe 'salir' para finalizar.\n")

ls_test_msj_memoria = []
while True:

    pregunta = input("Escribe tu mensaje: ").strip()

    if not pregunta:
        continue

    print(f"👤 Usuario: {pregunta}")

    if pregunta.lower() in ["salir", "exit", "quit"]:
        print("🤖 Agente: ¡Hasta luego! 👋\n")
        break

    respuesta = await agente.run(
        pregunta,
        session=sesion
    )

    ls_test_msj_memoria.append((pregunta, respuesta.text))

    print(f"🤖 Agente: {respuesta}\n")

# Recuérdame mi saldo, mi cuota y cuánto gano.
# ¿Qué palabras exactas usé en mi primer mensaje?
# ¿Qué te acabo de preguntar?

🤖 Agente: Hola 👋 ¿En qué tema de finanzas personales puedo ayudarte?
Escribe 'salir' para finalizar.

👤 Usuario: Recuérdame mi saldo, mi cuota y cuánto gano.
🤖 Agente: - **Saldo aproximado:** S/ 12,400  
- **Cuota mensual:** S/ 850  
- **Ingresos mensuales:** S/ 3,000

👤 Usuario: ¿Qué palabras exactas usé en mi primer mensaje?
🤖 Agente: Disculpa, no lo recuerdo

👤 Usuario: ¿Qué te acabo de preguntar?
🤖 Agente: Me preguntaste: “¿Qué palabras exactas usé en mi primer mensaje?”

👤 Usuario: exit
🤖 Agente: ¡Hasta luego! 👋

